In [ ]:
from pathlib import Path
import sys

# Resolve the project from the notebook's working directory (root or any subfolder).
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

import json
import pandas as pd
from fast_langdetect import detect

RAW_DIR = RAW_DEVGPT_DIR
PROCESSED_DIR = DATASET_DIR
if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"DevGPT raw snapshots were not supplied in this package: {RAW_DIR}. "
        "See README.md to obtain and place the snapshots."
    )
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
snapshot_dirs = sorted(folder for folder in RAW_DIR.iterdir() if folder.is_dir())
if not snapshot_dirs:
    raise FileNotFoundError(f"No DevGPT snapshot folders under {RAW_DIR}")
print(f"DevGPT snapshots: {len(snapshot_dirs)}")

In [ ]:
records = []

for snapshot_dir in snapshot_dirs:
    for json_file in sorted(snapshot_dir.glob("*.json")):
        try:
            with open(json_file, "r", encoding="utf-8") as file:
                data = json.load(file)
        except Exception as error:
            print(f"Error reading {json_file.name}: {error}")
            continue

        sources = data.get("Sources", [])

        for source in sources:
            sharings = source.get("ChatgptSharing", [])

            if not isinstance(sharings, list):
                sharings = [sharings]

            for sharing in sharings:
                if not isinstance(sharing, dict):
                    continue

                chatgpt_url = sharing.get("URL", "")
                conversations = sharing.get("Conversations", [])

                if not conversations:
                    continue

                for turn_index, turn in enumerate(conversations):
                    if not isinstance(turn, dict):
                        continue

                    prompt = turn.get("Prompt", "")

                    if not isinstance(prompt, str) or not prompt.strip():
                        continue

                    records.append(
                        {
                            "snapshot": snapshot_dir.name,
                            "source_file": json_file.name,
                            "chatgpt_url": str(chatgpt_url),
                            "turn_index": turn_index,
                            "prompt": prompt.strip(),
                            "answer": str(turn.get("Answer", "")),
                        }
                    )

df_all = pd.DataFrame(records)

df_all["turn_index"] = df_all["turn_index"].astype(int)

print(f"Total turns extracted: {len(df_all):,}")
print(f"Unique conversation URLs: {df_all['chatgpt_url'].nunique():,}")
print(f"Snapshots covered: {df_all['snapshot'].nunique()}")

print("\nTurns per snapshot:")
print(df_all["snapshot"].value_counts().sort_index())


In [ ]:
all_turns_path = INTERIM_DIR / "devgpt_all_turns.parquet"

df_all.to_parquet(all_turns_path, index=False)

print(f"Saved: {all_turns_path}")
print(f"Rows: {len(df_all):,}")


In [ ]:
turns_per_conversation = (
    df_all.groupby("chatgpt_url")["turn_index"]
    .max()
    .add(1)
    .reset_index(name="total_turns")
)

print(f"Unique conversation URLs: {len(turns_per_conversation):,}")

print("\nTurn-count statistics:")
print(turns_per_conversation["total_turns"].describe())

print("\nMost common conversation lengths:")
print(
    turns_per_conversation["total_turns"]
    .value_counts()
    .sort_index()
    .head(20)
)


In [ ]:
df_first = df_all[df_all["turn_index"] == 0].copy()

print(f"First-turn rows before deduplication: {len(df_first):,}")
print(f"Unique URLs before deduplication: {df_first['chatgpt_url'].nunique():,}")
print(f"Exact duplicated (URL, prompt) pairs: {df_first.duplicated(subset=['chatgpt_url', 'prompt']).sum():,}")


In [ ]:
# Remove exact repeated observations first
before_exact = len(df_first)

df_first = df_first.drop_duplicates(
    subset=["chatgpt_url", "prompt"],
    keep="first"
).copy()

print(f"Exact repeated observations removed: {before_exact - len(df_first):,}")

# Check whether any conversation URL still appears more than once
url_counts = df_first["chatgpt_url"].value_counts()
repeated_urls = url_counts[url_counts > 1]

print(f"URLs with multiple first-turn observations remaining: {len(repeated_urls):,}")

# Keep one deterministic observation per conversation URL
before_url = len(df_first)

df_first = df_first.drop_duplicates(
    subset=["chatgpt_url"],
    keep="first"
).copy()

df_first = df_first.reset_index(drop=True)

print(f"Additional rows removed by URL deduplication: {before_url - len(df_first):,}")
print(f"Final first-turn rows: {len(df_first):,}")
print(f"Final unique URLs: {df_first['chatgpt_url'].nunique():,}")


In [ ]:
assert df_first["prompt"].notna().all()
assert (df_first["prompt"].str.strip() != "").all()
assert df_first["chatgpt_url"].is_unique
assert (df_first["turn_index"] == 0).all()

first_turn_path = INTERIM_DIR / "devgpt_first_turn.parquet"

df_first.to_parquet(first_turn_path, index=False)

print(f"Saved: {first_turn_path}")
print(f"Rows: {len(df_first):,}")


In [ ]:
def detect_language(text):
    try:
        return detect(text[:500])[0]["lang"]
    except Exception:
        return "unknown"

df_first["lang"] = df_first["prompt"].apply(detect_language)

print("\nLanguage distribution:")
print(df_first["lang"].value_counts().head(15))


In [ ]:
df_english = df_first[df_first["lang"] == "en"].copy()
df_english = df_english.reset_index(drop=True)

print(f"First-turn conversations: {len(df_first):,}")
print(f"English prompts: {len(df_english):,}")
print(f"Non-English or unknown: {len(df_first) - len(df_english):,}")


In [ ]:
assert df_english["chatgpt_url"].is_unique
assert (df_english["turn_index"] == 0).all()
assert (df_english["lang"] == "en").all()
assert df_english["prompt"].notna().all()
assert (df_english["prompt"].str.strip() != "").all()

english_path = PROCESSED_DIR / "devgpt_first_turn_english.parquet"

df_english.to_parquet(english_path, index=False)

print(f"Saved: {english_path}")
print(f"Rows: {len(df_english):,}")
print(f"Unique URLs: {df_english['chatgpt_url'].nunique():,}")
print(f"Duplicate URLs: {df_english['chatgpt_url'].duplicated().sum():,}")
print(f"Duplicate prompt texts: {df_english['prompt'].duplicated().sum():,}")
print(f"Missing prompts: {df_english['prompt'].isna().sum():,}")
print("\nColumns:")
print(df_english.columns.tolist())
